In [0]:
%run ../../config/utils

In [0]:
import sys
sys.path.append("..")
sys.path.append("../..")

from lib.job_manager import load_config, split_config
from lib_etl.s3 import etl_input_data_validator
import lib_etl.validations_ETL as validations

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)

## Data availability check

In [0]:
source_path = data_paths["source"]["detail"]
recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
etl_input_data_validator(
    "source",
    recency_lookback_duration,
    data_paths,
    [
        "detail",
    ],
    spark
)

## Load source data

In [0]:
df = spark.read.csv(source_path, header=True, sep='|', quote='"', nullValue='?')
df.createOrReplaceTempView('df')

In [0]:
validations.validate_table(
        spark, "source", 'detail', config_validation, df, stats_etl_path
    )

## Save to delta table

In [0]:
spark.sql(f"""
    INSERT OVERWRITE {bronze_transaction_detail}
    SELECT
        PURCH_HDR_ID,
        PURCH_DTL_ID,
        PURCH_DT,
        GTIN_CD,
        ARTICLE_NBR,
        MC_CD,
        EXTENDED_PRC_AMT,
        EXTENDED_UNIT_PRC_AMT,
        SALES_QTY,
        SALES_UOM,
        QTY_IN_UNITS,
        NORMAL_PRC_AMT,
        NORMAL_UNIT_PRC_AMT,
        REDUCTION_AMT,
        SCANNED_VS_KEYED_IND,
        DISCOUNT_TYPE_CD,
        DISCOUNT_PURCH_DTL_ID,
        VOIDED_FLAG,
        VOIDED_PURCH_DTL_ID,
        RSN_CD,
        SALES_CTGRY_CD,
        RETURN_IND,
        REBATE_IND,
        OFFER_ID
    FROM df
""")